### KD (FKL) 为什么不是 RKL

> KD 用 FKL，本质上是把 Teacher 的输出当成“软标签数据分布”，让 Student 做监督学习式的 MLE；而 RKL 会变成 Student 自己采样、自己评估、自己调整的 on-policy 优化问题，梯度信号更稀疏、更高方差，也更容易 mode-seeking。

$$
\begin{split}
\theta_{\text{MLE}} &= \arg\max_{\theta} \mathbb{E}_{x \sim P_{\text{data}}} [\log Q_{\theta}(x)]
\end{split}
$$

- KD (FKL)：$\mathbb{E}_{a \sim p_{T}(\cdot|s)}$
    - 期望是基于 Teacher 的分布（或者是数据集的真实标签）产生的。
    - 输入一个数据 $s$，固定不动的 Teacher 吐出一组固定的概率分布。
    - Student 只需要针对这三个现成的数字去计算它自己的 $\log q_{\theta}$，然后直接做 Backpropagation（反向传播）即可。整个过程是确定性的，不需要任何随机采样。
- gradient
    - 对 Student logits 做 softmax cross entropy 时，梯度基本就是：$q_\theta(a|s)-p_T(a|s)$
    - 也就是说，每个 token 都有明确的方向：
        - Teacher 认为某 token 概率高，Student 给低了，就把它拉上来；
        - Teacher 认为某 token 概率低，Student 给高了，就把它压下去；

$$
\arg\max_{\theta} \mathbb{E}_{a \sim q_{\theta}(\cdot|s)} [\log p_{T}(a|s)]
$$
- $\mathbb{E}_{a \sim q_{\theta}(\cdot|s)}$
    - 期望是基于 Student 当前的分布产生的。
    - 为了计算这个损失函数，数据（动作 $a$）必须由 Student 自己通过当前的参数 $\theta$ 随机采样出来。
    -  Student 自己从当前策略 $q_\theta$ 里采样动作 $a$，然后 teacher 给这个动作一个分数
        -  $r(a)=\log p_T(a|s)$
        -  目标可以改造为：$\max_\theta \mathbb E_{a\sim q_\theta}[r(a)]=\sum_{a}q_\theta(a|s)\log p_T(a|s)$
        -  非常典型的 RL / policy optimization 形式：让当前 policy 多产生 reward 高的动作。
    - 期望的分布本身包含未知参数 $\theta$（即 $\nabla_{\theta} \mathbb{E}_{x \sim Q_{\theta}} [f(x)]$），无法直接把导数符号 $\nabla_{\theta}$ 放到期望符号里面去。如果想求导，就必须借助于强化学习中的 REINFORCE（Policy Gradient，策略梯度） 算法：
        - $\nabla_{\theta} \mathbb{E}_{a \sim q_{\theta}} [r(a)] = \mathbb{E}_{a \sim q_{\theta}} \left[ r(a) \cdot \nabla_{\theta} \log q_{\theta}(a|s) \right]$
            - score function trick：$\nabla_\theta p_\theta(x)=p_\theta(x)\nabla_\theta \log p_\theta(x)$
        - $\nabla_{\theta} \mathbb{E}_{a \sim q_{\theta}} [\log p_{T}(a|s)] = \mathbb{E}_{a \sim q_{\theta}} \left[ \log p_{T}(a|s) \cdot \nabla_{\theta} \log q_{\theta}(a|s) \right]$

#### gradient of FKL

$$
q_i = q_\theta(i \mid s)
= \frac{e^{z_i}}{\sum_k e^{z_k}}, \quad p_i = p_T(i \mid s)
$$

$$
\begin{aligned}
L(\theta)
&= -\sum_i p_i \log q_i = -\sum_i p_i \log \frac{e^{z_i}}{\sum_k e^{z_k}} \\[4pt]
&= -\sum_i p_i 
\left(
z_i - \log \sum_k e^{z_k}
\right) \\[4pt]
&= -\sum_i p_i z_i
+
\sum_i p_i \log \sum_k e^{z_k} \\[4pt]
&= -\sum_i p_i z_i
+
\left(\sum_i p_i\right)
\log \sum_k e^{z_k} \\[4pt]
&= -\sum_i p_i z_i
+
\log \sum_k e^{z_k}.
\end{aligned}
$$

$$
\begin{aligned}
\frac{\partial L}{\partial z_j}
&=
\frac{\partial}{\partial z_j}
\left(
-\sum_i p_i z_i
+
\log \sum_k e^{z_k}
\right) \\[4pt]
&=
-p_j
+
\frac{\partial}{\partial z_j}
\log \sum_k e^{z_k} \\[4pt]
&=
-p_j
+
\frac{1}{\sum_k e^{z_k}}
\frac{\partial}{\partial z_j}
\sum_k e^{z_k} \\[4pt]
&=
-p_j
+
\frac{e^{z_j}}{\sum_k e^{z_k}} \\[4pt]
&=
-p_j + q_j \\[4pt]
&=
q_j - p_j.
\end{aligned}
$$

$$
\boxed{
\nabla_z L=q_\theta(\cdot \mid s)-p_T(\cdot \mid s)
}
$$

### PG：score function trick (log derivative trick)

$$
\nabla_\theta p_\theta(x)=p_\theta(x)\nabla_\theta \log p_\theta(x)
$$

- $\nabla_\theta \log p_\theta(\tau)=\frac{\nabla_\theta p_\theta(\tau)}{p_\theta(\tau)}$，概率的相对变化率，也叫 score。它告诉策略：为了让刚刚采到的动作或轨迹更容易出现，参数应该往哪个方向移动。
- $J(\theta)=\mathbb{E}_{\tau \sim p_\theta(\tau)}[R(\tau)]$
    - $\tau = (s_0,a_0,s_1,a_1,\dots)$，表示一整条轨迹，$R(\tau)$ 是这条轨迹上的回报，问题就在于 轨迹 $\tau$ 是从策略 $\pi_\theta(a|s)$ 中采样出来的，采样过程本身依赖参数 $\theta$
    

$$
\begin{split}
\nabla_\theta J(\theta)&=\nabla_\theta \int p_\theta(\tau)R(\tau)d\tau\\
&=\int \nabla_\theta p_\theta(\tau)R(\tau)d\tau\\
&=\int p_\theta(\tau)\nabla_\theta \log p_\theta(\tau)R(\tau)d\tau\\
&=\mathbb{E}_{\tau \sim p_\theta(\tau)}[\nabla_\theta \log p_\theta(\tau)R(\tau)]
\end{split}
$$
- 将“对期望求导”转换为“求导数的期望”。
- 原本需要对“采样结果”求导，现在变成了对“采样这个结果的 log 概率”求导。
    - $\nabla_\theta p_\theta(\tau)\quad\Longrightarrow\quad p_\theta(\tau)\nabla_\theta \log p_\theta(\tau)\quad\Longrightarrow\quad \nabla_\theta \log p_\theta(\tau)$
- $\nabla_\theta J(\theta)=\sum_\tau \nabla_\theta p_\theta(\tau)R(\tau)$
    - “如果调整策略参数，哪些轨迹的概率会增加，哪些轨迹的概率会减少；然后根据这些轨迹的奖励，判断这次调整到底好不好。”
    - 大语言模型在几万词表中生成长序列中，对所有可能的轨迹 $\tau$ 进行积分 $\int (\dots) d\tau$ 是一个天文数字级别的运算，
    - 面对不可解的积分，需要蒙特卡洛采样（Monte Carlo Sampling）
    - 蒙特卡洛采样需要：被积函数必须能够被重写为一个基于概率分布的期望形式 $\mathbb{E}_{x \sim P(x)}[f(x)]$。
    - 公式中必须明确存在一个概率密度函数 $P(x)$（满足大于等于 0 且积分等于 1），计算机才能依据这个分布去“掷骰子”采样。
    - $\nabla_\theta p_\theta(\tau)$：绝非概率密度
        - 梯度，有正有负
        - $\nabla_\theta \int p_\theta(\tau) d\tau = \nabla_\theta (1)=0$
- $\nabla_\theta J(\theta)=\mathbb{E}_{\tau\sim p_\theta}[R(\tau)\nabla_\theta\log p_\theta(\tau)]$
    - 智能体只需按照当前的策略大脑 $p_\theta$ 在环境中玩几局（即进行采样），然后计算中括号内各项的平均值，就能得到极其逼近真实梯度的近似值。
    - $\nabla_\theta J(\theta) \approx \frac{1}{N} \sum_{i=1}^N \sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_{i,t}|s_{i,t}) R(\tau_i)$

### Monte Carlo Sampling 求期望

- 策略分布 $p_\theta(\tau)$：假设智能体的动作是一个连续的标量 $x$，策略分布是一个正态分布。参数 $\theta$ 就是这个分布的均值 $\mu$，方差固定为 1。即 $x \sim \mathcal{N}(\mu, 1)$。
- 黑盒奖励 $R(\tau)$：假设环境给出的打分规则很简单，动作的平方就是奖励，即 $R(x) = x^2$。
- 优化目标 $J(\theta)$：最大化期望奖励 $J(\mu) = \mathbb{E}_{x \sim \mathcal{N}(\mu, 1)}[x^2]$。
- 解析解形式：$J(\mu)
=
\mathbb{E}_{x\sim \mathcal{N}(\mu,1)}[x^2]
=\int_{-\infty}^{+\infty}x^2\frac{1}{\sqrt{2\pi}}\exp\left(-\frac{(x-\mu)^2}{2}\right)dx = \text{Var}(x) + (\mathbb{E}[x])^2 = 1 + \mu^2$
    - 对均值 $\mu$ 求导，直接可得真实的完美梯度：$$\nabla_\mu J(\mu) = 2\mu$$

- surrogate objective/loss

$$\mathbb{E} [ R(\tau) \log p_\theta(\tau) ]$$

In [13]:
import torch
from torch.distributions import Normal

mu = torch.tensor([2.0], requires_grad=True)

print(f"【解析解】真实的完美梯度应为: {2 * 2.0:.4f}\n")
# p_\theta
dist = Normal(mu, 1.0)

# 蒙特卡洛采样的样本数 (Batch Size)
N = 50000
# 与环境交互，采样 N 个动作 (相当于 \tau \sim p_\theta)
x = dist.sample((N,))

# 环境给出黑盒打分 R(\tau)，从外部世界拿来的客观常数，绝对不能对环境求导。
reward = (x ** 2).detach()

# 计算采样轨迹在当前策略下的对数概率 \log p_\theta(\tau)
log_prob = dist.log_prob(x)

# 构造 Score Function Trick 的代理目标函数 (Surrogate Objective)
# 理论公式: \mathbb{E} [ R(\tau) \log p_\theta(\tau) ]
# .mean: 期望的具象化
surrogate_obj = (reward * log_prob).mean()

surrogate_obj.backward()

print(f"【采样法】采样 {N} 次估算出的梯度为: {mu.grad.item():.4f}")

【解析解】真实的完美梯度应为: 4.0000

【采样法】采样 50000 次估算出的梯度为: 3.9955


### 理解 score：$\nabla_\theta \log p_\theta(\tau) = \frac{\nabla_\theta p_\theta(\tau)}{p_\theta(\tau)}$

- 如果不除以 $p_\theta(\tau)$（即仅看 $\nabla_\theta p_\theta$），优化器关注的是绝对概率的增量；除以 $p_\theta(\tau)$ 之后，优化器关注的则是相对概率（百分比）的增量。
- 假设智能体探索出了两个都能获得高分（$R=100$）的动作 A 和 B。
    - 动作 A 是一个极其罕见的神来之笔，当前策略下发生的概率仅为 1%。
    - 动作 B 是一个平庸但稳妥的操作，当前策略下发生的概率高达 50%。
- 如果目标是让这两个好动作的绝对概率都增加 1%（即 $\nabla_\theta p_\theta$ 相同）：
    - 对于 A，概率从 1% 提升到 2%，这意味着发生频率直接翻倍，策略参数 $\theta$ 需要发生剧烈的变动。
    - 对于 B，概率从 50% 提升到 51%，这只是基础上的微小波动，策略参数 $\theta$ 只需要极其微小的调整。

### PG -> PPO

下标是 $\tau \sim p_\theta(\tau)$。这意味着，用来计算这一步梯度的所有轨迹数据，必须严格由当前这一微秒的策略 $\theta$ 亲自采样得出（即 On-policy）。 一旦参数 $\theta$ 更新了一步变成了 $\theta'$，旧的采样数据就因为概率分布不再匹配而全部作废。这导致了极低的数据利用率。为了打破这种“采一次样，更一步新，然后全部扔掉”的困局，在如 PPO（Proximal Policy Optimization）或 GRPO 等强化学习后训练算法中，引入了重要性采样（Importance Sampling）：

$$
\mathbb{E}_{\tau \sim p_{old}} \left[ \frac{p_\theta(\tau)}{p_{old}(\tau)} R(\tau) \right]
$$